In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :memoryless

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [memoryless_model] Fitting chain 3 (tau=34)
[ Info: [memoryless] iter 1000/1000000 elapsed=3.9s, rate=0.267, mean=[1.071, 0.00250, 2.212], std=[0.0357, 0.001034, 0.4092] [ADAPT]
[ Info: [memoryless] iter 2000/1000000 elapsed=6.7s, rate=0.237, mean=[1.128, 0.00194, 2.527], std=[0.0630, 0.000902, 0.4172] [ADAPT]
[ Info: [memoryless] iter 3000/1000000 elapsed=8.8s, rate=0.224, mean=[1.150, 0.00173, 2.588], std=[0.0600, 0.000795, 0.3547] [ADAPT]
[ Info: [memoryless] iter 4000/1000000 elapsed=10.9s, rate=0.220, mean=[1.168, 0.00160, 2.660], std=[0.0600, 0.000722, 0.3348] [ADAPT]
[ Info: [memoryless] iter 5000/1000000 elapsed=13.0s, rate=0.219, mean=[1.176, 0.00154, 2.684], std=[0.0575, 0.000667, 0.3059] [ADAPT]
[ Info: [memoryless] iter 6000/1000000 elapsed=15.1s, rate=0.221, mean=[1.180, 0.00151, 2.718], std=[0.0544, 0.000621, 0.2907] [ADAPT]
[ Info: [memoryless] iter 7000/1000000 elapsed=17.2s, rate=0.222, mean=[1.182, 0.00149, 2.733], std=[0.0523, 0.000584, 0.2853] [ADAPT]
[ Info